# Lab 6 — Hosted Agent with RAG (Azure AI Search)

This lab extends the hosted agent with **Retrieval Augmented Generation**. The agent runs a search against an Azure AI Search index **before each model call** and grounds its answer in the retrieved documents.

You will:

1. Provision a small search index and seed it with documents (each with a unique *canary* token).
2. Scaffold and deploy a hosted agent that uses `AzureAISearchContextProvider`.
3. Grant the **hosted agent's own identity** read access to the search service.
4. Invoke the agent and verify grounding via the canary token.

> **Prerequisites:** everything from Lab 5 (Hosted Agent), plus an **Azure AI Search** service. Set `AZURE_SEARCH_ENDPOINT` and `AZURE_SEARCH_INDEX_NAME` in `.env`.


## 1. Provision the search index

Your identity needs **Search Service Contributor** (create index) and **Search Index Data Contributor** (upload docs) on the search service. RBAC can take a few minutes to propagate.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchableField, SearchFieldDataType, SearchIndex, SimpleField,
)

load_dotenv()

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
index_name = os.environ.get("AZURE_SEARCH_INDEX_NAME", "contoso-outdoors")
credential = DefaultAzureCredential()

DOCUMENTS = [
    {"id": "return-policy", "sourceName": "Contoso Outdoors Return Policy",
     "sourceLink": "https://contoso.com/policies/returns",
     "content": "Customers may return any item within 30 days of delivery. Items should be "
                "unused and include original packaging. Every accepted return ships back with "
                "a complimentary Contoso TrailRunner sticker pack, item code TR-CANARY-7821."},
    {"id": "shipping-guide", "sourceName": "Contoso Outdoors Shipping Guide",
     "sourceLink": "https://contoso.com/help/shipping",
     "content": "Standard shipping is free on orders over $50 and arrives in 3-5 business days. "
                "Use promo code SHIP-CANARY-4493 for a one-time free overnight upgrade."},
    {"id": "tent-care", "sourceName": "TrailRunner Tent Care Instructions",
     "sourceLink": "https://contoso.com/manuals/trailrunner-tent",
     "content": "Clean tent fabric with lukewarm water and non-detergent soap. Replacement "
                "waterproofing kits are stocked under SKU TENT-CANARY-9067."},
]

index = SearchIndex(
    name=index_name,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String, analyzer_name="standard.lucene"),
        SimpleField(name="sourceName", type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="sourceLink", type=SearchFieldDataType.String),
    ],
)

index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
try:
    index_client.get_index(index_name)
    print(f"Index '{index_name}' already exists; leaving schema as-is.")
except ResourceNotFoundError:
    index_client.create_index(index)
    print(f"Created index '{index_name}'.")

search_client = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)
results = search_client.merge_or_upload_documents(documents=DOCUMENTS)
print(f"Uploaded {sum(1 for r in results if r.succeeded)} / {len(DOCUMENTS)} documents.")


## 2. Scaffold the RAG agent source

The agent wires an `AzureAISearchContextProvider` into the `Agent`, so each turn is grounded in the index.

In [ ]:
import os
os.makedirs("lab6_agent/src/labs-hosted-agent-rag", exist_ok=True)
print("Created lab6_agent/src/labs-hosted-agent-rag")


In [ ]:
%%writefile lab6_agent/src/labs-hosted-agent-rag/main.py
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os

from agent_framework import Agent
from agent_framework.azure import AzureAISearchContextProvider
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()


async def main():
    credential = DefaultAzureCredential()
    search_provider = AzureAISearchContextProvider(
        source_id="azure_search_rag",
        endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
        index_name=os.environ["AZURE_SEARCH_INDEX_NAME"],
        credential=credential,
        mode="semantic",
        top_k=3,
    )
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )
    async with search_provider:
        agent = Agent(
            client=client,
            instructions=(
                "You are a helpful support specialist for Contoso Outdoors. "
                "Answer using the provided context and cite the source document."
            ),
            context_providers=[search_provider],
            default_options={"store": False},
        )
        server = ResponsesHostServer(agent)
        await server.run_async()


if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
%%writefile lab6_agent/src/labs-hosted-agent-rag/agent.yaml
# yaml-language-server: $schema=https://raw.githubusercontent.com/microsoft/AgentSchema/refs/heads/main/schemas/v1.0/ContainerAgent.yaml
kind: hosted
name: labs-hosted-agent-rag
description: |
  An Agent Framework agent with RAG over Azure AI Search.
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "1"
  memory: 2Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: gpt-4.1
  - name: AZURE_SEARCH_ENDPOINT
    value: REPLACE_WITH_YOUR_SEARCH_ENDPOINT
  - name: AZURE_SEARCH_INDEX_NAME
    value: contoso-outdoors


In [ ]:
%%writefile lab6_agent/src/labs-hosted-agent-rag/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY . user_agent/
WORKDIR /app/user_agent
RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
EXPOSE 8088
CMD ["python", "main.py"]


In [ ]:
%%writefile lab6_agent/src/labs-hosted-agent-rag/requirements.txt
agent-framework
agent-framework-azure-ai-search
agent-framework-foundry-hosting


In [ ]:
%%writefile lab6_agent/azure.yaml
# yaml-language-server: $schema=https://raw.githubusercontent.com/Azure/azure-dev/main/schemas/v1.0/azure.yaml.json
name: labs-hosted-agent-rag
requiredVersions:
  extensions:
    azure.ai.agents: '>=0.1.0-preview'
services:
  labs-hosted-agent-rag:
    project: src/labs-hosted-agent-rag
    host: azure.ai.agent
    language: docker
    docker:
      remoteBuild: true
    config:
      container:
        resources:
          cpu: "1"
          memory: 2Gi
      startupCommand: python main.py


## 3. Deploy

Edit `lab6_agent/src/labs-hosted-agent-rag/agent.yaml` and set `AZURE_SEARCH_ENDPOINT` to your search endpoint, then deploy in a terminal (Docker required):

```bash
cd lab6_agent
azd ai agent init        # bind to existing project, model gpt-4.1
azd deploy
```


## 4. Grant the hosted agent's identity read access — the key gotcha

A hosted agent queries Azure AI Search as its **own dedicated agent identity** (via `DefaultAzureCredential` inside the container) — **not** your user and **not** the project managed identity. So after the first deploy you must grant *that* identity **Search Index Data Reader** on the search service, or every query returns `403 Forbidden`.

The cell below reads the agent's instance + blueprint identities from the version definition and grants the role to both (the instance identity is what the container uses; the blueprint identity is stable across versions).

In [ ]:
import json
import subprocess
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import requests

load_dotenv()

AGENT_NAME = "labs-hosted-agent-rag"
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
# Full ARM resource id of your search service (subscriptions/.../searchServices/<name>)
SEARCH_SERVICE_ID = os.environ.get("AZURE_SEARCH_SERVICE_ID", "<paste-search-service-resource-id>")

token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token
url = f"{project_endpoint}/agents/{AGENT_NAME}/versions/1?api-version=2025-11-15-preview"
ver = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=60).json()

ids = {
    "instance": ver["instance_identity"]["principal_id"],
    "blueprint": ver["blueprint"]["principal_id"],
}
print("Agent identities:", json.dumps(ids, indent=2))

for label, pid in ids.items():
    cmd = [
        "az", "role", "assignment", "create",
        "--assignee-object-id", pid,
        "--assignee-principal-type", "ServicePrincipal",
        "--role", "Search Index Data Reader",
        "--scope", SEARCH_SERVICE_ID,
    ]
    print(f"Granting {label} ({pid}) ...")
    subprocess.run(cmd, shell=True)

## 5. Invoke and verify grounding

Ask about the return policy. A grounded answer includes the **canary token** `TR-CANARY-7821`, which proves the response came from the retrieved document and not the model's training data.

> RBAC propagation can take a few minutes — if you get `status: failed` right after granting, wait and re-run.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv()

AGENT_NAME = "labs-hosted-agent-rag"
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token
url = f"{project_endpoint}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1"

resp = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"input": "What is your return policy? Include any item codes mentioned.", "store": False},
    timeout=120,
)
data = resp.json()
print("status:", data.get("status"))
for item in data.get("output", []):
    for part in item.get("content", []):
        if part.get("type") == "output_text":
            print(part["text"])

If the reply contains `TR-CANARY-7821`, RAG is working end-to-end. 🎉

You've now built both **prompt agents** and **hosted agents** (basic and RAG) using the latest Foundry + Agent Framework SDKs.